In [7]:
import sys

print(sys.version)

# XGBOOST WAS DONE IN PYTHON 3.12.13

3.12.13 | packaged by Anaconda, Inc. | (main, Mar 19 2026, 20:12:32) [Clang 20.1.8 ]


In [10]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_excel('CLEANED_DATASET_THESIS_FINAL.xlsx') # CHANGE DIRECTORY TO RIGHT PATH

df["TIME_PERIOD"] = pd.to_datetime(df["TIME_PERIOD"].astype(str).str.replace("-M", "-"), format="%Y-%m").dt.to_period("M")

new_rows = []

for country, group in df.groupby('COUNTRY'):
    group = group.sort_values('TIME_PERIOD').copy()

    temp = group[['COUNTRY', 'TIME_PERIOD', 'OBS_VALUE', 'GPR', 'ANNUALIZED_VOLATILITY']].copy()

    temp['OBS_VALUE'] = pd.to_numeric(temp['OBS_VALUE'], errors='coerce')
    temp['GPR'] = pd.to_numeric(temp['GPR'], errors='coerce')

    temp['LAG_3'] = temp['OBS_VALUE'].shift(3)
    temp['ROLL_MEAN_3'] = temp['OBS_VALUE'].rolling(3).mean()
    temp['ROLL_MEAN_12'] = temp['OBS_VALUE'].rolling(12).mean()
    temp['LOG_RETURN'] = np.log(temp['OBS_VALUE'] / temp['OBS_VALUE'].shift(1))

    temp['TARGET'] = temp['OBS_VALUE'].shift(-1)
    temp = temp.dropna().reset_index(drop=True)

    preds = []
    acts = []

    vol_preds = []
    vol_acts = []

    start_train_size = 24

    features = [
        'OBS_VALUE',
        'GPR',
        'LAG_3',
        'ROLL_MEAN_3',
        'ROLL_MEAN_12',
        'LOG_RETURN']

    for i in range(start_train_size, len(temp)):
        
        train = temp.iloc[:i]
        test = temp.iloc[i:i+1]

        X_train = train[features].astype(float)
        y_train = train['TARGET']

        X_test = test[features].astype(float)
        y_test = test['TARGET']

        model = XGBRegressor(n_estimators=25, learning_rate=0.05, max_depth=3, random_state=42, n_jobs=1, tree_method='hist')
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)[0]

        preds.append(y_pred)
        acts.append(y_test.values[0])

        obs_value = test['OBS_VALUE'].values[0]

        log_ret_act = np.log(y_test.values[0] / obs_value)
        log_ret_pred = np.log(y_pred / obs_value)

        vol_acts.append(log_ret_act)
        vol_preds.append(log_ret_pred)

    mae_pred = mean_absolute_error(acts, preds)
    mse_pred = mean_squared_error(acts, preds)

    vol_act_series = pd.Series(vol_acts)
    vol_pred_series = pd.Series(vol_preds)

    vol_act = vol_act_series.rolling(12).std() * np.sqrt(12)
    vol_pred = vol_pred_series.rolling(12).std() * np.sqrt(12)

    mask = vol_act.notna()

    mae_vol = mean_absolute_error(vol_act[mask], vol_pred[mask])
    mse_vol = mean_squared_error(vol_act[mask], vol_pred[mask])

    last_row = temp.iloc[-1]

    X_next = last_row[features].to_frame().T.astype(float)
    prediction = model.predict(X_next)[0]

    new_time = group['TIME_PERIOD'].iloc[-1] + 1

    prev_value = last_row['TARGET']
    per_change = (prediction - prev_value) / prev_value

    log_return = np.log(prediction / prev_value)

    last_11 = temp['LOG_RETURN'].dropna().iloc[-11:]
    last_12 = pd.concat([last_11, pd.Series([log_return])])
    st_dev = last_12.std()

    annualized_vol = st_dev * np.sqrt(12)

    new_row = {
        'COUNTRY': country,
        'TIME_PERIOD': new_time,
        'PRED_VAL_XGB': prediction,
        'PER_CHANGE': per_change,
        'LOG_RETURN': log_return,
        'ST_DEV': st_dev,
        'ANNUALIZED_VOLATILITY': annualized_vol,
        'XGB_PRED_MAE': mae_pred,
        'XGB_PRED_MSE': mse_pred,
        'XGB_VOL_MAE': mae_vol,
        'XGB_VOL_MSE': mse_vol
    }

    new_rows.append(new_row)

df_new = pd.DataFrame(new_rows)
df_updated = pd.concat([df, df_new], ignore_index=True)
df_updated = df_updated.sort_values(['COUNTRY', 'TIME_PERIOD'])
df_updated.to_excel('XGB_EXPW_Updated_Clean_DF.xlsx', index=False)


In [ ]:
df_metrics = pd.read_excel('XGB_EXPW_Updated_Clean_DF.xlsx')

print(df_metrics['XGB_PRED_MAE'].describe())

print(df_metrics['XGB_PRED_MSE'].describe())

print(df_metrics['XGB_VOL_MAE'].describe())

print(df_metrics['XGB_VOL_MSE'].describe())